In [2]:
import requests, pandas as pd, sqlite3
from bs4 import BeautifulSoup as bs
from sodapy import Socrata

In [3]:
url = "https://data.cityofchicago.org/resource/v6vf-nfxy.json"
response = requests.get(url)
print("Our response code is:", response.status_code) 
# check response code to make sure it works 

Our response code is: 200


### Testing with the token

In [4]:
# I am citing this from Chicago data portal 311 API documentation and socrata API code snippets (Chicago Open Data portal uses Socrata)
# On the code snippets, it has instructions and code on how to access the API data with token
# using the original template, I am building on the given template
# "https://dev.socrata.com/foundry/data.cityofchicago.org/v6vf-nfxy"
# "https://dev.socrata.com/docs/queries/page"


token = "iAZp9mxjcw3dt8SUNugTPYJbS" 
client = Socrata("data.cityofchicago.org", token, timeout = 60)

DATASET_ID = "v6vf-nfxy"

SELECT_COLS = ",".join(["sr_number",
    "sr_type",
    "sr_short_code",
    "status",
    "origin",
    "created_department",
    "owner_department",
    "created_date",
    "last_modified_date",
    "closed_date",
    "latitude",
    "longitude",
    "zip_code",
    "community_area",
    "ward",
    "duplicate",
    "legacy_record"
])

BASE_WHERE = """
created_date >= '{start}' AND created_date < '{end}'
AND duplicate = false AND legacy_record = false
"""

def fetch_311(start, end, limit=50000):
    """Fetch ALL rows for [start, end) using Socrata paging."""
    offset = 0
    out = []

    where = BASE_WHERE.format(start=start, end=end)

    while True:
        rows = client.get(
            DATASET_ID,
            select=SELECT_COLS,
            where=where,
            order="created_date ASC, sr_number ASC",
            limit=limit,
            offset=offset
        )
        if not rows:
            break

        out.append(pd.DataFrame.from_records(rows))

        if len(rows) < limit:
            break

        offset += limit

    if out:
        return pd.concat(out, ignore_index=True)
    return pd.DataFrame()


### Pulling data in chunks

In [5]:
import time

def fetch_all(where, limit=20000, max_retries=5):
    offset = 0
    frames = []

    while True:
        attempt = 0
        while True:
            try:
                rows = client.get(
                    DATASET_ID,
                    select=SELECT_COLS,
                    where=where,
                    order="created_date ASC, sr_number ASC",
                    limit=limit,
                    offset=offset
                )
                break  
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise  

                wait = 2 * attempt 
                print(f"Timeout, trying...")
                time.sleep(wait)

        if not rows:
            break

        frames.append(pd.DataFrame.from_records(rows))

        if len(rows) < limit:
            break

        offset += limit
        time.sleep(0.2) 

    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

In [7]:
where_jan2023 = BASE_WHERE.format(
    start="2023-01-01T00:00:00.000",
    end="2023-02-01T00:00:00.000"
)

df_jan = fetch_all(where_jan2023, limit=50000)
print("Jan 2023 rows:", len(df_jan))
df_jan.head()

Jan 2023 rows: 123412


,sr_number,sr_type,sr_short_code,status,origin,owner_department,created_date,last_modified_date,latitude,longitude,community_area,ward,duplicate,legacy_record,closed_date,created_department,zip_code
0,SR23-00000001,Water Lead Test Kit Request,WCA2,Open,DWM,DWM - Department of Water Management,2023-01-01T00:01:17.000,2023-01-01T00:30:34.000,41.99731200094083,-87.79661100000192,10,41,False,False,NaN,NaN,NaN
1,SR23-00000002,Water Lead Test Kit Request,WCA2,Completed,DWM,DWM - Department of Water Management,2023-01-01T00:02:15.000,2023-02-27T14:55:07.000,41.99731200094083,-87.79661100000192,10,41,False,False,2023-02-27T14:55:07.000,NaN,NaN
2,SR23-00000003,Water Lead Test Kit Request,WCA2,Open,DWM,DWM - Department of Water Management,2023-01-01T00:02:39.000,2023-01-01T00:30:34.000,41.99731200094083,-87.79661100000192,10,41,False,False,NaN,NaN,NaN
3,SR23-00000004,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,2023-01-01T00:11:26.000,2023-01-01T00:30:34.000,41.87183400094043,-87.6798450000019,28,28,False,False,2023-01-01T00:11:27.000,311 City Services,60612
4,SR23-00000005,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,2023-01-01T00:11:51.000,2023-01-01T00:30:34.000,41.87183400094043,-87.6798450000019,28,28,False,False,2023-01-01T00:11:52.000,311 City Services,60612


- Missing rate of zipcode is too high, we have decided to switch from using zipcode to community area and ward number. 

In [8]:
df_test = pd.read_csv("data/raw_data/311_2023_01.csv")
print(df_test["zip_code"].isna().mean())
print(df_test["zip_code"].value_counts(dropna=False).head(10))

0.38914368132758564
zip_code
60612.0    51225
NaN        48025
60666.0    21540
60649.0       90
60630.0       87
60620.0       81
60617.0       80
60629.0       78
60641.0       76
60647.0       75
Name: count, dtype: int64


In [ ]:
def next_month(y, m):
    if m == 12:
        return (y + 1, 1)
    return (y, m + 1)

for y in [2021, 2022, 2023, 2024, 2025]:
    for m in range(1, 13):
        y2, m2 = next_month(y, m)

        start = f"{y}-{m:02d}-01T00:00:00.000"
        end   = f"{y2}-{m2:02d}-01T00:00:00.000"
        tag   = f"{y}_{m:02d}"

        where_m = BASE_WHERE.format(start=start, end=end)
        df_m = fetch_all(where_m, limit=20000)

        print(tag, "rows:", len(df_m))
        df_m.to_csv(f"data/raw_data/311_{tag}.csv", index=False)

### Data Cleaning

In [10]:
import glob

files = sorted(glob.glob("data/raw_data/311_*.csv"))
dfs = [pd.read_csv(f) for f in files]
df_311 = pd.concat(dfs, ignore_index=True)

print("total:", len(df_311))
df_311.head()

total: 8867645


,sr_number,sr_type,sr_short_code,status,origin,created_department,owner_department,created_date,last_modified_date,closed_date,latitude,longitude,zip_code,community_area,ward,duplicate,legacy_record
0,SR21-00000001,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:00:09.000,2021-01-01T00:31:03.000,2021-01-01T00:00:09.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False
1,SR21-00000002,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:00:56.000,2021-01-01T00:31:03.000,2021-01-01T00:00:56.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False
2,SR21-00000004,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:01:19.000,2021-01-01T00:31:22.000,2021-01-01T00:01:19.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False
3,SR21-00000005,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:02:09.000,2021-01-01T00:31:30.000,2021-01-01T00:02:09.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False
4,SR21-00000006,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:02:17.000,2021-01-01T00:31:03.000,2021-01-01T00:02:17.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False


In [7]:
def add_basic_metrics_cols(df):
    df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
    df["closed_date"]  = pd.to_datetime(df["closed_date"], errors="coerce")

    df["year"] = df["created_date"].dt.year

    df["is_closed"] = df["closed_date"].notna()

    df["ttc_hours"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
    df.loc[df["ttc_hours"] < 0, "ttc_hours"] = pd.NA  

    return df


In [14]:
df_311.to_csv("data/311_2021_to_2025_clean.csv", index=False)

In [8]:
df = pd.read_csv("data/311_2021_to_2025_clean.csv") 

df["community_area"] = pd.to_numeric(df["community_area"], errors="coerce") # make sure community area is numberic so it is easier to use for merging with school profile
df = df.dropna(subset=["community_area"]).copy()
df["community_area"] = df["community_area"].astype(int)

df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
df["closed_date"]  = pd.to_datetime(df["closed_date"], errors="coerce")
df["year"] = df["created_date"].dt.year

df["is_closed"] = df["closed_date"].notna()
df["ttc_hours"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
df.loc[df["ttc_hours"] < 0, "ttc_hours"] = pd.NA

final_metrics = (
    df.groupby(["community_area", "year", "ward"])
      .agg(
          n_requests=("sr_number", "size"),
          n_closed=("is_closed", "sum"),
          share_closed=("is_closed", "mean"),
          median_ttc_hours=("ttc_hours", "median"),
          p75_ttc_hours=("ttc_hours", lambda s: s.quantile(0.75)),
      )
      .reset_index()
)

final_metrics.to_csv("data/311_metrics.csv", index=False)

In [10]:
print(len(df)) # check length
print(df["created_date"].min(), "to", df["created_date"].max()) # check time period 
print("missing community_area:", df["community_area"].isna().mean()) # check missing rate
print("missing closed_date:", df["closed_date"].isna().mean()) # ceck missing rate for closed_date for request

final_metrics.sort_values("n_requests", ascending=False).head(10)


8858595
2021-01-01 00:00:09 to 2025-12-31 23:59:32
missing community_area: 0.0
missing closed_date: 0.013374694294072592


,community_area,year,ward,n_requests,n_closed,share_closed,median_ttc_hours,p75_ttc_hours
585,28,2025,28.0,667862,667443,0.999373,0.0,0.000278
578,28,2024,28.0,661937,661586,0.999470,0.0,0.000278
559,28,2021,28.0,637367,637320,0.999926,0.0,0.000278
572,28,2023,28.0,635590,635466,0.999805,0.0,0.000278
565,28,2022,28.0,613181,613132,0.999920,0.0,0.000278
1345,76,2025,41.0,366926,366805,0.999670,0.0,0.000278
1343,76,2024,41.0,345383,345366,0.999951,0.0,0.000278
1339,76,2022,41.0,326975,326964,0.999966,0.0,0.000278
1337,76,2021,41.0,318274,318268,0.999981,0.0,0.000278
1341,76,2023,41.0,309663,309645,0.999942,0.0,0.000278


### Reading the new compiled and cleaned file

In [11]:
df2 = pd.read_csv("data/311_metrics.csv")  

In [12]:
df2.columns

Index(['community_area', 'year', 'ward', 'n_requests', 'n_closed',
       'share_closed', 'median_ttc_hours', 'p75_ttc_hours'],
      dtype='str')

### EDA 